# Modelo Analítico Parametrizado · Da base bruta à decisão
**Projeto:** _nome do projeto_ · **Tema:** _tema da análise_ · **Base:** _arquivo/tabela e período_ · **Autor:** Pedro Henrique Costa da Silva

> **Como usar**
> 1. Preencha a seção 1 (problema).
> 2. Edite **somente a célula de parâmetros (seção 2)**: fonte, colunas, KPIs, regras de qualidade e análises.
> 3. Rode tudo (`Run All`). A lógica fica no pacote `modelo_analitico/`, que não precisa de edição.
> 4. Escreva a leitura de cada análise e a conclusão.
>
> **Dica de escrita:** em células Markdown, escreva valores em reais como `R\$ 9,8 mi` (com a barra). Dois `$` soltos no mesmo parágrafo fazem o Jupyter/VS Code tratar o trecho entre eles como fórmula matemática, o que deixa o texto com a letra diferente.

## 1. Definição do problema de negócio
**Contexto.** _Quem é a organização, qual o setor/processo e como ele gera (ou perde) resultado._

**Problema.** _A pergunta de decisão e quem a fez._

**Objetivos.**
1. Tratar a base **sem alterar a fonte original** (todo o tratamento ocorre em cópia) e documentar cada decisão.
2. Medir os KPIs essenciais por segmento e ao longo do tempo.
3. Resumir a principal conclusão em 1 gráfico e entregar insights acionáveis com uma decisão recomendada.

**Perguntas-chave.**
- P1. Que inconsistências existem na base e como foram tratadas?
- P2. Qual o desempenho dos KPIs por segmento?
- P3. Como os KPIs evoluem no tempo?
- P4. Onde está a maior oportunidade?

## 2. Parâmetros (única célula a editar a cada novo projeto)

In [ ]:
from modelo_analitico import Config, Modelo, tabela_kpis

def ajustes(d):
    """Correções específicas do caso (opcional). Recebe o DataFrame tratado e o devolve."""
    d["comissao_pct"] = d["comissao_pct"].round(2)
    d["comissao_corrigida"] = (d["valor_venda"] * d["comissao_pct"]).round(2)          # recalcula a comissão
    d["flag_ajuste"] = (d["comissao"] - d["comissao_corrigida"]).abs() > 0.011         # auditoria: linhas corrigidas
    return d

cfg = Config(
    # ── Fonte: arquivo, URL, banco (sqlite:///x.db, postgresql://...) ou DataFrame ──
    fonte="Case_B_Base_Lotes_Lance_Certo.xlsx",
    tipo="auto",                       # auto | csv | xlsx | json | jsonl | parquet | feather | pickle | sql
    leitura={"sheet_name": "Lotes"},   # extras do pandas: sep=";", encoding="latin-1", sheet_name=...
    consulta=None,                     # SQL quando a fonte é banco, ex.: "SELECT * FROM lotes"

    # ── Esquema (nomes DEPOIS de renomear) ──
    renomear={"lance_inicial_R$": "lance_inicial", "valor_venda_R$": "valor_venda", "comissao_R$": "comissao"},
    id="id_lote", data="data_leilao", freq="M",       # freq: D | W | M | Q | Y
    dimensoes=["categoria", "canal", "uf"],
    sucesso=("vendido", ["S"]),        # (coluna, valores de sucesso) → cria "flag"; None se não se aplica
    numericas=[],                      # colunas de texto a converter em número, ex.: "1.234,56"
    periodo=("2026-01-01", "2026-09-30"),             # período esperado (None para ignorar)

    # ── KPIs ──
    moeda="R$",
    medidas={                          # nome: (coluna, agregação, formato)   formato: num | moeda | pct
        "lotes":    ("id_lote", "count", "num"),
        "vendidos": ("flag", "sum", "num"),
        "gmv":      ("valor_venda", "sum", "moeda"),
        "comissão": ("comissao_corrigida", "sum", "moeda"),
    },
    razoes={                           # nome: (numerador, denominador, formato)
        "conversão":         ("vendidos", "lotes", "pct"),
        "take_rate":         ("comissão", "gmv", "pct"),
        "ticket_médio":      ("gmv", "vendidos", "moeda"),
        "comissão_por_lote": ("comissão", "lotes", "moeda"),
    },
    kpi_principal="comissão",          # ordena rankings e base da "participação"

    # ── Qualidade ──
    regras={                           # nome: expressão que identifica linhas INVÁLIDAS (esperado: 0)
        "Não vendido com valor de venda": "(vendido == 'N') & valor_venda.notna()",
        "Vendido sem valor de venda":     "(vendido == 'S') & valor_venda.isna()",
        "Venda abaixo do lance inicial":  "valor_venda < lance_inicial",
        "Lance inicial nulo ou <= 0":     "lance_inicial.isna() | (lance_inicial <= 0)",
        "Comissão ≠ valor × percentual":  "(comissao - valor_venda * comissao_pct.round(2)).abs() > 0.011",
    },
    outlier_cols=["lance_inicial", "valor_venda"], outlier_grupo="categoria",   # outliers medidos DENTRO de cada grupo
    conferir=["valor_venda", "comissao", "comissao_corrigida"],                  # somas comparadas antes × depois
    ajustes=ajustes,

    # ── Análises (nomes definidos acima) ──
    dim_a="categoria", dim_b="canal",
    metricas_rank=["comissão", "conversão", "comissão_por_lote"],
    metricas_tempo=["gmv", "comissão", "conversão", "take_rate"],
    metrica_cruz="conversão",

    pasta_saida="saidas", seed=42,
)
m = Modelo(cfg)

## 3. KPIs (tabela gerada a partir dos parâmetros)
A referência de cada KPI é a média da própria base, salvo se houver meta.

In [ ]:
tabela_kpis(cfg)

## 4. Carga e exploração dos dados
A fonte é **apenas lida**. Todo o tratamento acontece em cópia.

In [ ]:
m.carregar()

In [ ]:
m.explorar()

## 5. Diagnóstico de qualidade (antes da limpeza)
Nada é alterado nesta etapa. As regras de negócio vêm do parâmetro `regras`. Os outliers são medidos dentro de cada grupo, porque bases com itens de preços muito diferentes parecem ter outliers quando se olha tudo junto.

In [ ]:
m.diagnosticar()

**Leitura e decisão.** _Descreva o que foi encontrado (duplicatas, regras violadas, outliers) e a decisão tomada para cada item (corrigir, remover ou manter, e por quê)._

## 6. Limpeza, atributos e verificação (em cópia)

In [ ]:
m.tratar()

## 7. Análises
Cada análise segue **pergunta → código → gráfico → leitura**. Para trocar de análise, altere os parâmetros ou chame o método com outra dimensão, por exemplo `m.ranking("uf")`.

### 7.1 Visão geral

In [ ]:
m.geral()

### 7.2 Ranking por dimensão principal

In [ ]:
m.ranking()

**Leitura.** _Responda a pergunta com os destaques acima (em valores monetários escreva `R\$`)._

### 7.3 Comparação entre grupos da dimensão B

In [ ]:
m.comparar()

**Leitura.** _A diferença é estatisticamente relevante? Lembre: significância não prova causalidade; o perfil dos grupos pode diferir._

### 7.4 Gráfico principal: dimensão A × dimensão B

In [ ]:
m.cruzar(titulo="")      # escreva um título que diga a conclusão

### 7.5 Evolução no tempo

In [ ]:
m.evolucao()

**Leitura.** _Há tendência, sazonalidade ou um período fora do padrão?_

### 7.6 Último período contra a média dos anteriores

In [ ]:
m.ultimo_vs_anteriores()

**Leitura.** _O desvio é concentrado em algum segmento? Atenção a amostras pequenas._

### 7.7 Análise livre
_Use `m.kpis("coluna")` para qualquer agrupamento e `m.mostrar(nome, tabela)` para exibir e registrar a tabela na exportação._

In [ ]:
m.demais_dimensoes()      # ranking das dimensões que não são A nem B

## 8. Conclusão e decisão recomendada
**Perfil geral.** _Resuma com os números da seção 7.1._

**Insights acionáveis**
1. **_Insight_.** _Evidência (números) e por que importa._
2. **_Insight_.** _Evidência e por que importa._
3. **_Insight_.** _Evidência e por que importa._

**Qualidade dos dados.** _Impacto das correções (tabela da seção 6)._

| Item | Definição |
|---|---|
| **Critério de decisão** | _Como priorizar_ |
| **Ação** | _O que fazer_ |
| **Responsável (sugestão)** | _Quem_ |
| **Prazo (sugestão)** | _Quando_ |
| **Indicador de acompanhamento** | _KPI e meta_ |

## 9. Exportação das saídas

In [ ]:
m.exportar()

## 10. Checklist final
- [ ] Fonte original não foi alterada (tratamento só em cópia)
- [ ] Pergunta de negócio respondida
- [ ] Números conferidos (verificações da seção 6 passaram)
- [ ] Inconsistências documentadas com a decisão tomada
- [ ] Gráficos com título que diz a conclusão
- [ ] Decisão com critério, ação, responsável, prazo e indicador